使用模型，获取不同场景数据

### coef set

基于目标系数，政策时期，构建分段函数  

In [1]:
from path_config import *
import mygeo as mg
import myfunction as mf

import warnings
warnings.filterwarnings('ignore')

from joblib import Parallel, delayed

import random
import pandas as pd
from scipy.stats import boxcox


In [2]:
# 路径
dir_sw_raw_high   = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_high\raw"  
dir_sw_pfas_high  = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_high\pfas"  

dir_sw_raw_low    = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_low\raw"   
dir_sw_pfas_low   = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_low\pfas"

dir_sw_raw_us = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_us\raw"   
dir_sw_pfas_us = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_us\pfas"

dir_sw_raw_base   = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\raw"  
dir_sw_pfas_base  = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\pfas"  

dir_sw_raw_2020  = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw"  
dir_sw_pfas_2020 = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\pfas"


dir_lr_raw_high = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high\raw"
dir_lr_pfas_high = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high\pfas"
dir_lr_merge_high = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high\merge"

dir_lr_raw_low = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low\raw"
dir_lr_pfas_low = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low\pfas"
dir_lr_merge_low = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low\merge"

dir_lr_raw_us = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us\raw"
dir_lr_pfas_us = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us\pfas"
dir_lr_merge_us = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us\merge"

dir_lr_raw_2020 = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw"
dir_lr_merge_2020 = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\merge"
dir_lr_pfas_2020  = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\pfas"  

dir_lr_raw_base = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\raw"
dir_lr_merge_base = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\merge" 
dir_lr_pfas_base  = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\pfas"  


In [3]:
# 场景参数
import pprint

df_meta = pd.read_csv(path_file + 'meta_data.csv', encoding='utf-8-sig')
df_coef = df_meta[['var_name','coef_down','coef_up']].dropna()
factor_dict_base = {
    row['var_name']: {
        'coef_down': 1,
        'coef_up': 1
    }
    for _, row in df_coef.iterrows()
}

factor_dict = {
    row['var_name']: {
        'coef_down': float(row['coef_down']),
        'coef_up': float(row['coef_up'])
    }
    for _, row in df_coef.iterrows()
}
scenario_factors_range = {'bl':factor_dict_base, 'rs1': factor_dict,'rs2': factor_dict,'us': factor_dict}
pprint.pprint(scenario_factors_range['rs1'])

{'GDP': {'coef_down': 0.008, 'coef_up': 0.02},
 'SWD_LDF_CH4': {'coef_down': 0.8, 'coef_up': 0.9},
 'WWT_CH4': {'coef_down': 0.75, 'coef_up': 0.95},
 'clothing': {'coef_down': 0.35, 'coef_up': 0.45},
 'fluorite_consumption': {'coef_down': 0.1, 'coef_up': 0.15},
 'manufacturing': {'coef_down': 0.016, 'coef_up': 0.04},
 'paper_consumption': {'coef_down': 0.1, 'coef_up': 0.15},
 'potential_contamination': {'coef_down': 0.8, 'coef_up': 0.9},
 'wrap_consumption': {'coef_down': 0.25, 'coef_up': 0.35}}


In [4]:
int_rdm = 202603
str_describe_lr = 'lr'
str_describe_sw = 'sw'
save_file = True

list_pfas = ['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 
             'PFOS', 'FOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', 
             'HFPO-DA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS']


df_po = pd.read_excel(path_file + inf_file, sheet_name="po_treat")
list_pfas_all = df_po['posname'].tolist()
list_pfas_all_lc = df_po[df_po['po_chain']==1]['posname'].tolist()
list_pfas_all_sc = df_po[df_po['po_chain']==0]['posname'].tolist()
print(len(list_pfas_all_lc))
print(len(list_pfas_all_sc))
list_list_pfas = [list_pfas_all, list_pfas_all_lc, list_pfas_all_sc]

df_treat = df_po[['posname', 'start_year']][df_po['start_year']>2000]
dict_treat = dict(zip(df_treat['posname'], df_treat['start_year']))
print(dict_treat)
list_treat = list(dict_treat.keys())


df_meta = pd.read_csv(path_file + meta_file)
list_bio_var = df_meta[df_meta['var_select_bio'] == 1]['var_name'].tolist()
list_water_var = df_meta[df_meta['var_select_w'] == 1]['var_name'].tolist()

df_geo_raw = pd.read_csv(path_part0_pre + 'geo_global_raw_co2.csv')
# df_geo_raw = mg.convert_temp(df_geo_raw, "K", "C")
# df_geo_lr_var = df_geo_raw[df_geo_raw.columns.intersection(list_bio_var)]
# df_geo_sw_var = df_geo_raw[df_geo_raw.columns.intersection(list_water_var)]

list_po = df_meta[(df_meta['var_select_all'] == 1)&(df_meta['var_type3'] == 2)]['var_name'].tolist()
if "posname" in list_po:
    list_po.remove("posname")
dict_inf_po = {"posname":list_po}

df_sp_cluster_treat = pd.read_csv(path_part0_pre + "sp_rep.csv")

def safe_pfas(name):
    return name.replace(':', '-').replace(' ', '-').replace('/', '-')

list_pfas_all_safe     = [safe_pfas(x) for x in df_po['posname'].tolist()]
list_pfas_all_lc_safe  = [safe_pfas(x) for x in df_po[df_po['po_chain'] == 1]['posname'].tolist()]
list_pfas_all_sc_safe  = [safe_pfas(x) for x in df_po[df_po['po_chain'] == 0]['posname'].tolist()]
print(len(list_pfas_all_safe))
print(len(list_pfas_all_lc_safe))
print(len(list_pfas_all_sc_safe))


list_pfas_all_safe_lr = ['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 
             'PFOS', 'FOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', 
             'HFPO-DA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS']

list_pfas_all_lc_safe_lr = ['PFNA', 'PFOA', 'PFOS', 'PFDA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 
                            'PFTrDA', 'FOSA', 'PFHxS','PFNS', 'PFDS', 'PFHpS']
list_pfas_all_sc_safe_lr = ['PFBS', 'PFHpA', 'PFHxA', 'HFPO-DA', 'PFBA', 'PFPeA', 'PFPeS']


df_epi_norm = pd.read_csv(r'C:\Users\dell\OneDrive\file\csv\scenario\epi_norm_1x1.csv')
treat_value = ['log', 'zscore']
df_result_loaded = pd.read_pickle(path_part0_temp + "geo_transform_params.pkl")
list_list_pfas = [list_pfas_all, list_pfas_all_lc_safe, list_pfas_all_sc_safe]


37
23
{'HFPO-DA': 2009, 'ADONA': 2008}
60
37
23


In [6]:


def process_data_list_parm(path_part3, path_part0_match, str_describe, match_filename, scaling_method="minmax"):
    """
    按照逻辑处理数据并返回两个结果列表：
    list_input: [处理后的数据集DataFrame, 最优参数表DataFrame, 选中特征list, 最优模型名]
    list_inv_parm: 如果 scaling_method='minmax' -> [最小值, 最大值, Box-Cox λ值]
                   如果 scaling_method='zscore' -> [均值, 标准差, Box-Cox λ值]
    """
    best_params = pd.read_csv(path_part3 + 'ml_cv_best.csv')
    best_params = best_params.sort_values(by='score', ascending=False)
    best_model = best_params.iloc[0]['model']
    print("Best model:", best_model)
    selected_features = pd.read_csv(
        path_part3 + str_describe + "_rfecv_features_" + best_model + "cv.csv"
    )
    selected_features = selected_features[selected_features["Rank"] == 1]["Feature"].values
    df_raw = pd.read_csv(path_part0_match + match_filename)
    df_data_raw = df_raw.copy()
    print(df_data_raw['posname'].unique().tolist())
    print(len(df_data_raw['posname'].unique().tolist()))
    df_data_raw['value'], lam = boxcox(df_data_raw['value'])
    print(f'Box-Cox Lambda: {lam}')
    print(f"box-cox max value:{df_data_raw['value'].max()}, box-cox min value:{df_data_raw['value'].min()}")
    if scaling_method.lower() == "minmax":
        max_val = df_data_raw['value'].max()
        min_val = df_data_raw['value'].min()
        print(f"[MinMax] Max: {max_val}, Min: {min_val}")
        df_data_raw['value'] = (df_data_raw['value'] - min_val) / (max_val - min_val)
        list_inv_parm = [min_val, max_val, lam]
    elif scaling_method.lower() == "zscore":
        mean_val = df_data_raw['value'].mean()
        std_val = df_data_raw['value'].std(ddof=0)
        print(f"[ZScore] Mean: {mean_val}, Std: {std_val}")
        df_data_raw['value'] = (df_data_raw['value'] - mean_val) / std_val
        print(f'zscore max value:{df_data_raw["value"].max()},zscore min value:{df_data_raw["value"].min()}')
        list_inv_parm = [mean_val, std_val, lam]
    else:
        raise ValueError("scaling_method must be 'minmax' or 'zscore'.")
    df_data_raw['year'] = (df_data_raw['year'] - 2000) / (2020 - 2000)
    list_input = [df_data_raw, best_params, selected_features, best_model]
    return list_input, list_inv_parm


list_input_sw, list_inv_parm_sw = process_data_list_parm(
    path_part3=path_part3_sw,
    path_part0_match=path_part0_match,
    str_describe=str_describe_sw,
    match_filename="sw_match_geo_auto_zscore.csv",
    scaling_method="zscore"
)

list_input_lr, list_inv_parm_lr = process_data_list_parm(
    path_part3=path_part3_lr,
    path_part0_match=path_part0_match,
    str_describe=str_describe_lr,
    match_filename="lr_refcv_match.csv",
    scaling_method="zscore"
)


Best model: LGBM
['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 'PFOS', 'FOSA', 'EtFOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', '6:2 Cl-PFESA', '6:2 FTSA', '6:2 diPAP', '6:2/8:2 diPAP', '6:8 PFPIA', 'EtFOSAA', 'EtFOSE', 'FOSAA', 'HFPO-DA', 'MeFOSA', 'MeFOSAA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS', 'PFECHS', 'PFHxDA', '4:2 FTSA', '8:2 FTSA', 'PFODA', 'PFPeDA', '8:2 diPAP', '10:2 FTSA']
39
Box-Cox Lambda: 0.008311325940076589
box-cox max value:11.03752441815122, box-cox min value:-10.610172456262653
[ZScore] Mean: -1.2075369943593433, Std: 2.9563841404384954
zscore max value:4.141904715634944,zscore min value:-3.1804511914709086
Best model: LGBM
['PFBS', 'PFDA', 'PFDS', 'PFDoDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFOS', 'PFTeDA', 'PFTrDA', 'PFUnDA', 'PFHpS', 'PFNA', 'PFOA', 'PFPeA', 'FOSA', 'PFPeDA', 'PFBA', 'PFHxDA', 'PFODA', '6:2 Cl-PFESA', 'EtFOSAA', '4:2 FTSA', '6:2 FTSA', '8:2 FTSA', 'EtFOSA', 'HFPO-DA', 'MeFOSA', 'MeFOSAA', 'PFNS', 'PFPeS', 'EtFOSE', '6:8 PFPI

### water hist and base

In [ ]:
def raw_geo_treat(df_geo, treat_value=False, df_result=None, list_remove=['lon_grid', 'lat_grid', 'year']):
    df_base = df_geo.copy()
    if treat_value:
        _, df_base_treat = mg.transform_geo_data(
                            df_geo_data=df_base,
                            list_remove=list_remove,
                            df_result=df_result
                            )
    if len(treat_value) > 1:
        df_base_treat, _ = mg.normalize_geo_from_csv(df_base_treat, treat_value[1])
    return df_base_treat


SAVE_DIR = r"F:\User_file\wyy\SPDB\part3_forecast\sw"
os.makedirs(SAVE_DIR, exist_ok=True)
def run_scenario(seed, scenario_key, scenario_name, output_dir, year_filter):
    df_geo_all = mg.apply_random_factors(
        df_geo_raw, scenario_factors_range[scenario_key],
        df_epi_norm, seed, scenario=scenario_name
    )
    df_geo_treat_all = raw_geo_treat(df_geo_all, treat_value, df_result_loaded)
    df_geo_filtered = df_geo_treat_all[year_filter(df_geo_treat_all)].copy()
    df_2025 = df_geo_filtered[df_geo_filtered['year'] == 2025]
    if not df_2025.empty:
        save_path = os.path.join(SAVE_DIR, f"{scenario_name}_seed{seed}.csv")
        df_2025.to_csv(save_path, index=False)

def process_single_seed(seed, scenario_idx):
    """处理单个seed的单个场景"""
    scenarios = [
        ("bl", "baseline",  dir_sw_raw_2020, lambda df: df['year'] < 2025),
        ("bl", "baseline",  dir_sw_raw_base, lambda df: df['year'] >= 2025),
        ("rs1","rs1",       dir_sw_raw_high, lambda df: df['year'] >= 2025),
        ("rs2","rs2",       dir_sw_raw_low,  lambda df: df['year'] >= 2025),
        # ("us", "us",        dir_sw_raw_us,   lambda df: df['year'] >= 2025),
    ]
    
    key, name, out_dir, year_filter = scenarios[scenario_idx]
    run_scenario(seed, key, name, out_dir, year_filter)


mc_test_time = 100
n_scenarios = 4
shared_seeds = True  # True: 五个场景共享同一组种子; False: 每个场景独立随机种子

random.seed(2025)

if shared_seeds:
    # 所有场景共用同一组 mc_test_time 个种子
    base_seeds = random.sample(range(20251226), mc_test_time)
    scenario_seeds = [base_seeds for _ in range(n_scenarios)]
else:
    # 每个场景独立抽取种子
    all_seeds = random.sample(range(20251226), mc_test_time * n_scenarios)
    scenario_seeds = [all_seeds[i*mc_test_time : (i+1)*mc_test_time] for i in range(n_scenarios)]

Parallel(n_jobs=8)(
    delayed(process_single_seed)(seed, scenario_idx)
    for scenario_idx, seeds in enumerate(scenario_seeds)
    for seed in seeds
)

[None, None, None, None, None, None, None, None, None, None]

In [25]:
import pandas as pd
import os

SAVE_DIR = r"F:\User_file\wyy\SPDB\part3_forecast\sw"
seed = 2775025

scenarios = {
    "baseline": f"baseline_seed{seed}.csv",
    "rs1": f"rs1_seed{seed}.csv",
    "rs2": f"rs2_seed{seed}.csv",
    "us": f"us_seed{seed}.csv",
}

# 读取所有文件
dfs = {name: pd.read_csv(os.path.join(SAVE_DIR, fname)) for name, fname in scenarios.items()}

# 数值列（排除坐标和year）
exclude_cols = ["lat_grid", "lon_grid", "year"]
value_cols = [c for c in dfs["baseline"].columns if c not in exclude_cols]

# 统计指标
stats = ["mean", "25%", "50%", "75%"]

records = []
for col in value_cols:
    for scenario, df in dfs.items():
        desc = df[col].describe()
        records.append({
            "column": col,
            "scenario": scenario,
            "mean": desc["mean"],
            "q25": desc["25%"],
            "q50": desc["50%"],
            "q75": desc["75%"],
        })

df_long = pd.DataFrame(records)

# 转成宽表：每列每个统计量对应4个场景
df_wide = df_long.pivot(index="column", columns="scenario", values=["mean", "q25", "q50", "q75"])
df_wide.columns = [f"{stat}_{scen}" for stat, scen in df_wide.columns]
df_wide = df_wide.reset_index()

# 列排序：按场景分组
ordered_cols = ["column"]
for stat in ["mean", "q25", "q50", "q75"]:
    for scen in scenarios.keys():
        ordered_cols.append(f"{stat}_{scen}")

df_wide = df_wide[ordered_cols]

out_path = os.path.join(SAVE_DIR, f"scenario_comparison_seed{seed}.csv")
df_wide.to_csv(out_path, index=False)
print(f"saved to {out_path}")

saved to F:\User_file\wyy\SPDB\part3_forecast\sw\scenario_comparison_seed2775025.csv


In [7]:
def raw_geo_treat(df_geo, treat_value=False, df_result=None, list_remove=['lon_grid', 'lat_grid', 'year']):
    df_base = df_geo.copy()
    if treat_value:
        _, df_base_treat = mg.transform_geo_data(
                            df_geo_data=df_base,
                            list_remove=list_remove,
                            df_result=df_result
                            )
    if len(treat_value) > 1:
        df_base_treat, _ = mg.normalize_geo_from_csv(df_base_treat, treat_value[1])
    return df_base_treat


def run_scenario(seed, scenario_key, scenario_name, output_dir, year_filter):
    """针对单个seed运行一个场景"""
    df_geo_all = mg.apply_random_factors(
        df_geo_raw, scenario_factors_range[scenario_key],
        df_epi_norm, seed, scenario=scenario_name
    )
    df_geo_treat_all = raw_geo_treat(df_geo_all, treat_value, df_result_loaded)
    df_geo_filtered = df_geo_treat_all[year_filter(df_geo_treat_all)].copy()
    float_cols = df_geo_filtered.select_dtypes(include=['float64']).columns
    df_geo_filtered[float_cols] = df_geo_filtered[float_cols].round(6)
    mg.sw_forecast_and_save_nc(
        list_input_sw, list_inv_parm_sw, df_geo_filtered,
        output_dir, list_list_pfas, dict_inf_po, seed, treat_value
    )

def process_single_seed(seed, scenario_idx):
    """处理单个seed的单个场景"""
    scenarios = [
        ("bl", "baseline",  dir_sw_raw_2020, lambda df: df['year'] < 2025),
        ("bl", "baseline",  dir_sw_raw_base, lambda df: df['year'] >= 2025),
        ("rs1","rs1",       dir_sw_raw_high, lambda df: df['year'] >= 2025),
        ("rs2","rs2",       dir_sw_raw_low,  lambda df: df['year'] >= 2025),
        ("us", "us",        dir_sw_raw_us,   lambda df: df['year'] >= 2025),
    ]
    
    key, name, out_dir, year_filter = scenarios[scenario_idx]
    run_scenario(seed, key, name, out_dir, year_filter)


mc_test_time = 100
n_scenarios = 5
shared_seeds = True  # True: 五个场景共享同一组种子; False: 每个场景独立随机种子

random.seed(2026)

if shared_seeds:
    # 所有场景共用同一组 mc_test_time 个种子
    base_seeds = random.sample(range(20260300), mc_test_time)
    scenario_seeds = [base_seeds for _ in range(n_scenarios)]
else:
    # 每个场景独立抽取种子
    all_seeds = random.sample(range(20260300), mc_test_time * n_scenarios)
    scenario_seeds = [all_seeds[i*mc_test_time : (i+1)*mc_test_time] for i in range(n_scenarios)]

Parallel(n_jobs=10)(
    delayed(process_single_seed)(seed, scenario_idx)
    for scenario_idx, seeds in enumerate(scenario_seeds)
    for seed in seeds
)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [8]:
from joblib import Parallel, delayed


tasks = [
    (dir_sw_raw_2020, dir_sw_pfas_2020),
    (dir_sw_raw_base, dir_sw_pfas_base),
    (dir_sw_raw_high, dir_sw_pfas_high),
    (dir_sw_raw_low,  dir_sw_pfas_low),
    (dir_sw_raw_us,   dir_sw_pfas_us),
]


Parallel(n_jobs=10)(
    delayed(mg.from_seed_get_pfas_dask)(
        dir_seed=dir_seed,
        dir_pfas=dir_pfas,
        str_describe="sw",
        chunk_seed=1,    
        chunk_lat=60,    
        chunk_lon=120,
        approx=True
    )
    for dir_seed, dir_pfas in tasks
)


[None, None, None, None, None]

### fish

In [9]:
def raw_geo_treat(df_geo, treat_value=False, df_result=None, list_remove=['lon_grid', 'lat_grid', 'year']):
    df_base = df_geo.copy()
    if treat_value:
        _, df_base_treat = mg.transform_geo_data(
                            df_geo_data=df_base,
                            list_remove=list_remove,
                            df_result=df_result
                            )
    if len(treat_value) > 1:
        df_base_treat, _ = mg.normalize_geo_from_csv(df_base_treat, treat_value[1])
    return df_base_treat



In [ ]:
list_pfas_all_safe_lr_short = ['PFBS', 'PFDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 
             'PFOS', 'FOSA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 'PFTrDA', 
             'HFPO-DA', 'PFBA', 'PFDS', 'PFHpS', 'PFNS', 'PFPeA', 'PFPeS']

list_pfas_all_lc_safe_lr_short = ['PFNA', 'PFOA', 'PFOS', 'PFDA', 'PFDoDA', 'PFTeDA', 'PFUnDA', 
                            'PFTrDA', 'FOSA', 'PFHxS','PFNS', 'PFDS', 'PFHpS']
list_pfas_all_sc_safe_lr_short = ['PFBS', 'PFHpA', 'PFHxA', 'HFPO-DA', 'PFBA', 'PFPeA', 'PFPeS']

list_list_pfas_lr_short = [list_pfas_all_safe_lr_short, list_pfas_all_lc_safe_lr_short, list_pfas_all_sc_safe_lr_short]


PFAS_GROUPS = {
    'list1': ['PFOA'],
    'list2': ['PFTeDA'],
    'list3': ['PFPeA', 'PFHxA', 'PFHpA'],
    'list4': ['PFUnDA', 'PFTrDA', 'PFDoDA'],
    'list5': ['PFDS'],
    'list6': ['PFBA'],
    'list7': ['PFHxS'],
    'list8': ['PFBA'],
    'list9': ['PFHxS'],
    'list10': ['PFBS', 'PFNS'],
    'list11': ['4:2 FTSA', '6:2 FTSA', 'HFPO-DA'],
    'list12': ['PFDA', 'PFNA'],
    'list13': ['PFOS'],
    'list14': ['FOSA', 'EtFOSAA', 'EtFOSA', 'MeFOSA', 'MeFOSAA']
    }



FEATURE_REMOVE_SINGLE = [
    'po_carbon', 'solubility', 'density', 'log_pKa', 'log_Koil_w',
    'po_m_w', 'log_D5_5', 'po_chain', 'log_Px'
]




In [11]:
# ── 工具函数：根据 PFAS 名称查找所属分组 ─────────────────────
import xarray as xr
import numpy as np
from lightgbm import LGBMRegressor
from scipy.stats import boxcox
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def custom_inv(y, list_inv_param, list_treat_value):
    """
    多种逆变换处理
    参数:
        y: 已转换后的值（标量、数组、Series）
        list_inv_param: 三个参数
            zscore: (mean, std, lam)
            minmax: (min, max, lam)
        list_treat_value: 一个或两个元素
            - 如果一个元素: 直接进行box-cox逆变换
            - 两个元素: 第二个元素必须是"zscore"或"minmax"
    返回:
        原始尺度的值
    """
    a = list_inv_param[0]  # mean 或 min
    b = list_inv_param[1]  # std 或 max
    lam = list_inv_param[2]

    if len(list_treat_value) == 1:
        y_bc = y  # 只做 Box-Cox 逆变换
    else:
        transform_type = list_treat_value[1].lower()
        if transform_type == "zscore":
            # 反标准化
            mean = a
            std = b
            y_bc = y * std + mean
        elif transform_type == "minmax":
            # 反归一化
            min_val = a
            max_val = b
            y_bc = y * (max_val - min_val) + min_val
        else:
            raise ValueError("list_treat_value[1] 必须是 'zscore' 或 'minmax'")

    # 反Box-Cox
    if lam == 0:
        return np.exp(y_bc)
    else:
        return np.power(lam * y_bc + 1, 1 / lam)


def get_pfas_group(pfas_name, group_dict=PFAS_GROUPS):
    for group_name, pfas_list in group_dict.items():
        if pfas_name in pfas_list:
            return group_name
    # 若不在任何分组中，归入默认组（用全量特征训练）
    return None
# ── 核心函数1：分组训练 ───────────────────────────────────────
def train_grouped_models(df_raw, best_params, selected_features_all, best_model, seed,
                         group_dict=PFAS_GROUPS):
    """
    按 PFAS 分组分别训练模型。
    返回:
        grouped_models : dict
            {group_name: {"model": trained_model, "features": feature_list}}
        default_model  : dict  (针对不在任何分组中的 PFAS 使用全量训练集)
            {"model": trained_model, "features": selected_features_all}
    """
    df_data = df_raw.copy()
    model_params = best_params[best_params["model"] == best_model].iloc[0]
    if pd.isna(model_params["max_depth"]) or str(model_params["max_depth"]).lower() == 'none':
        param_max_depth = None
    else:
        param_max_depth = int(float(model_params["max_depth"]))
    def _build_model():
        if best_model == 'RF':
            return RandomForestRegressor(
                max_depth=param_max_depth,
                min_samples_leaf=int(model_params["min_samples_leaf"]),
                min_samples_split=int(model_params["min_samples_split"]),
                n_estimators=int(model_params["n_estimators"]),
                random_state=seed
            )
        elif best_model == 'XGBR':
            return XGBRegressor(
                max_depth=param_max_depth,
                learning_rate=model_params["learning_rate"],
                min_child_weight=int(model_params["min_child_weight"]),
                gamma=int(model_params["gamma"]),
                n_estimators=int(model_params["n_estimators"]),
                random_state=seed,
                subsample=0.8,
                n_jobs=1
            )
        elif best_model == 'LGBM':
            return LGBMRegressor(
                max_depth=param_max_depth,
                learning_rate=model_params["learning_rate"],
                min_child_samples=int(model_params["min_child_samples"]),
                num_leaves=int(model_params["num_leaves"]),
                n_estimators=int(model_params["n_estimators"]),
                random_state=seed,
                subsample=0.8,
                subsample_freq=1,
                n_jobs=1
            )
        else:
            raise ValueError(f"Unsupported model: {best_model}")
    grouped_models = {}
    for group_name, group_pfas in group_dict.items():
        df_sub = df_data[df_data['posname'].isin(group_pfas)].copy()
        # 单 PFAS 组移除无区分度的性质特征
        if len(group_pfas) == 1:
            features = [f for f in selected_features_all if f not in FEATURE_REMOVE_SINGLE]
            print(f"[{group_name}] Single-PFAS group → removed PFAS-property features. "
                  f"Features: {len(features)}")
        else:
            features = selected_features_all.copy()
            print(f"[{group_name}] Multi-PFAS group → keep all features. "
                  f"Features: {len(features)}")
        df_sub = df_sub.dropna(subset=features)
        n = len(df_sub)
        print(f"[{group_name}] Training samples: {n}, PFAS: {group_pfas}")
        if n < 5:
            print(f"[{group_name}] WARNING: Too few samples (<5), skip training.")
            grouped_models[group_name] = None
            continue
        X_train = df_sub[features]
        y_train = df_sub['value']
        m = _build_model()
        m.fit(X_train, y_train)
        grouped_models[group_name] = {"model": m, "features": features}
        print(f"[{group_name}] Model trained.")
    # 默认模型（全量数据，处理不在任何分组中的 PFAS）
    df_default = df_data.dropna(subset=selected_features_all)
    m_default = _build_model()
    m_default.fit(df_default[selected_features_all], df_default['value'])
    default_model = {"model": m_default, "features": selected_features_all}
    print("[default] Full-data model trained.")
    return grouped_models, default_model
# ── 核心函数2：分组预测并保存 nc ─────────────────────────────
def lr_forecast_and_save_nc(list_model_build, list_inv_parm, df_input, save_path_lr_nc, save_path_sw_nc,
                             list_list_pfas, dict_inf_po, df_sp_cluster_treat, seed, treat_value,
                             list_inv_parm_sw, lam_sw=0.0083,
                             group_dict=PFAS_GROUPS):
    """
    使用分组训练的模型进行 LR 预测，并将结果保存为 .nc 文件。
    与原版的差异：
        - 调用 train_grouped_models 替换 train_forecast_model
        - 每条 PFAS 预测时，根据 get_pfas_group() 选取对应的子模型和特征集
    """
    df_raw, best_params, selected_features_all, best_model = list_model_build
    a = list_inv_parm_sw[0]  # mean 或 min
    b = list_inv_parm_sw[1]  # std 或 max
    list_all_pfas, list_lc_pfas, list_sc_pfas = list_list_pfas
    # ── 训练所有分组模型 ──
    grouped_models, default_model = train_grouped_models(
        df_raw, best_params, selected_features_all, best_model, seed, group_dict
    )
    # ── 遍历物种类别 ──
    for j in range(0, 5):
    #for j in [0, 4]:
        all_pfas_data = {}
        for pfas in list_all_pfas:
            safe_pfas = pfas.replace(':', '-').replace(' ', '-').replace('/', '-')
            # 根据 PFAS 选取对应模型与特征
            group_name = get_pfas_group(pfas, group_dict)
            if group_name is not None and grouped_models.get(group_name) is not None:
                model_bundle = grouped_models[group_name]
            else:
                print(f"[WARNING] {pfas} not in any group or group model unavailable → use default model")
                model_bundle = default_model
            model = model_bundle["model"]
            selected_features = model_bundle["features"]
            # ── 读取水体预测结果 ──
            sw_forecast = xr.open_dataset(os.path.join(save_path_sw_nc, f"sw_{safe_pfas}.nc"))
            df_sw_forecast = sw_forecast.to_dataframe().reset_index()
            sw_forecast.close()
            df_sw_forecast = df_sw_forecast.rename(columns={'mean': 'sw_value'})
            # ── 初始化预测输入数据 ──
            df_forecast_data = df_input.copy()
            df_forecast_data['posname'] = pfas
            # ── 添加额外信息 ──
            df_forecast_data = mf.append_inf(df_forecast_data, dict_inf_po, treat_value)
            # ── 合并水体预测值 ──
            df_forecast_data = pd.merge(
                df_forecast_data,
                df_sw_forecast[['year', 'lat', 'lon', 'sw_value']],
                left_on=['year', 'lat_grid', 'lon_grid'],
                right_on=['year', 'lat', 'lon'],
                how='left'
            )
            # ── 年份归一化 + Box-Cox 变换水体值 ──
            original_year = df_forecast_data['year'].copy()
            df_forecast_data['year'] = (df_forecast_data['year'] - 2000) / (2020 - 2000)
            df_forecast_data['sw_value'] = boxcox(df_forecast_data['sw_value'], lmbda=lam_sw)
            if isinstance(treat_value, (list, tuple)) and len(treat_value) > 1:
                if treat_value[1] == 'minmax':
                    df_forecast_data['sw_value'] = (df_forecast_data['sw_value'] - a) / (b - a)
                elif treat_value[1] == 'zscore':
                    df_forecast_data['sw_value'] = (df_forecast_data['sw_value'] - a) / b
            # ── 物种特征 ──
            sp_data = df_sp_cluster_treat[df_sp_cluster_treat['index'] == j][
                ['sp_length', 'sp_weight', 'sp_troph']].iloc[0]
            df_forecast_data['sp_length'] = sp_data['sp_length']
            df_forecast_data['sp_weight'] = sp_data['sp_weight']
            df_forecast_data['sp_troph'] = sp_data['sp_troph']
            # ── 固定器官类型 ──
            df_forecast_data['organ_muscle'] = 1
            df_forecast_data['organ_liver'] = 0
            # ── 特征选取（使用分组对应的特征集）──
            X_forecast = df_forecast_data[selected_features]
            # ── 模型预测 ──
            y_pred = model.predict(X_forecast)
            df_forecast_data['value'] = y_pred
            # ── 逆 Box-Cox 变换 ──
            df_forecast_data['lr_value'] = custom_inv(df_forecast_data['value'], list_inv_parm, treat_value)
            df_forecast_data['year'] = original_year
            # ── 转成 xarray ──
            df_pfas = df_forecast_data[['lon_grid', 'lat_grid', 'year', 'lr_value']].copy()
            df_pfas = df_pfas.rename(columns={'lon_grid': 'lon', 'lat_grid': 'lat'})
            ds_pfas = df_pfas.set_index(['year', 'lat', 'lon']).to_xarray()
            all_pfas_data[safe_pfas] = ds_pfas['lr_value']
        # ── 合并所有 PFAS 数据 ──
        ds_combined = xr.Dataset(all_pfas_data)
        # 注意：list_all_pfas 中的名称需要与 safe_pfas 对应，这里统一做转换
        def _safe(name):
            return name.replace(':', '-').replace(' ', '-').replace('/', '-')
        ds_combined['value']    = sum([ds_combined[_safe(v)] for v in list_all_pfas])
        ds_combined['lc_value'] = sum([ds_combined[_safe(v)] for v in list_lc_pfas])
        ds_combined['sc_value'] = sum([ds_combined[_safe(v)] for v in list_sc_pfas])
        for var in ds_combined.data_vars:
            ds_combined[var] = ds_combined[var].round(4)
        # ── 保存 nc 文件 ──
        os.makedirs(save_path_lr_nc, exist_ok=True)
        save_path = os.path.join(save_path_lr_nc, f"lr_forecast_{seed}_{j}.nc")
        ds_combined.to_netcdf(save_path)
        ds_combined.close()
        print(f"Saved: {save_path}")

In [12]:
lam_sw = 0.008311324

def run_scenario(seed, scenario_key, scenario_name, output_dir_lr, input_sw_dor, year_filter):
    """针对单个seed运行一个场景"""
    df_geo_all = mg.apply_random_factors(
        df_geo_raw, scenario_factors_range[scenario_key],
        df_epi_norm, seed, scenario=scenario_name
    )
    df_geo_treat_all = raw_geo_treat(df_geo_all, treat_value, df_result_loaded)
    df_geo_filtered = df_geo_treat_all[year_filter(df_geo_treat_all)].copy()
    float_cols = df_geo_filtered.select_dtypes(include=['float64']).columns
    df_geo_filtered[float_cols] = df_geo_filtered[float_cols].round(6)
    # df_geo_filtered = df_geo_filtered[(df_geo_filtered['year'] >= 2020) & (df_geo_filtered['year'] <= 2050)]
    lr_forecast_and_save_nc(list_input_lr, list_inv_parm_lr, df_geo_filtered, output_dir_lr, input_sw_dor,
                            list_list_pfas_lr_short, dict_inf_po, df_sp_cluster_treat, seed, treat_value, list_inv_parm_sw, lam_sw)


def process_single_seed(seed, scenario_idx):
    """处理单个seed的单个场景"""
    scenarios = [
        ("bl", "baseline",  dir_lr_raw_2020, dir_sw_pfas_2020, lambda df: df['year'] < 2025),
        ("bl", "baseline",  dir_lr_raw_base, dir_sw_pfas_base, lambda df: df['year'] >= 2025),
        ("rs1", "rs1",      dir_lr_raw_high, dir_sw_pfas_high, lambda df: df['year'] >= 2025),
        ("rs2", "rs2",      dir_lr_raw_low,  dir_sw_pfas_low,  lambda df: df['year'] >= 2025),
        ("us",  "us",       dir_lr_raw_us,   dir_sw_pfas_us,   lambda df: df['year'] >= 2025),
    ]

    key, name, out_dir_lr, out_dir_sw, year_filter = scenarios[scenario_idx]
    run_scenario(seed, key, name, out_dir_lr, out_dir_sw, year_filter)


mc_test_time = 20
n_scenarios = 5
shared_seeds = True  # True: 五个场景共享同一组种子; False: 每个场景独立随机种子

random.seed(2026)

if shared_seeds:
    base_seeds = random.sample(range(20260300), mc_test_time)
    scenario_seeds = [base_seeds for _ in range(n_scenarios)]
else:
    all_seeds = random.sample(range(20260300), mc_test_time * n_scenarios)
    scenario_seeds = [all_seeds[i*mc_test_time : (i+1)*mc_test_time] for i in range(n_scenarios)]

Parallel(n_jobs=10)(
    delayed(process_single_seed)(seed, scenario_idx)
    for scenario_idx, seeds in enumerate(scenario_seeds)
    for seed in seeds
)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [13]:
from joblib import Parallel, delayed

params = [
    (dir_lr_raw_2020, dir_lr_pfas_2020),
    (dir_lr_raw_base, dir_lr_pfas_base),
    (dir_lr_raw_high, dir_lr_pfas_high),
    (dir_lr_raw_low,  dir_lr_pfas_low),
    (dir_lr_raw_us,   dir_lr_pfas_us),
]

Parallel(n_jobs=10)(
    delayed(mg.from_seed_get_pfas_dask)(
        dir_seed=seed_dir,
        dir_pfas=pfas_dir,
        str_describe="lr",
        chunk_seed=1,   
        chunk_lat=60,   
        chunk_lon=120,
        approx=True
    )
    for seed_dir, pfas_dir in params
)


[None, None, None, None, None]

### 额外处理

把没有水的地区去除

In [14]:
import os
import numpy as np
import xarray as xr


def only_water_nc(water_file, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    water_ds = xr.open_dataset(water_file)
    water_mask = water_ds['water']  # DataArray，值为0或1
    for fname in os.listdir(input_dir):
        if fname.endswith(".nc"):
            fpath = os.path.join(input_dir, fname)
            
            ds = xr.open_dataset(fpath)
        
            mask = water_mask
            mask = mask.sel(lon=ds['lon'], lat=ds['lat'])
            for var in ds.data_vars:
                if np.issubdtype(ds[var].dtype, np.number):
                    ds[var] = xr.where(mask == 1, ds[var], np.nan)
        
            outpath = os.path.join(output_dir, fname)
            ds.to_netcdf(outpath)
            ds.close()

    print("处理完成！")


In [15]:
from path_config import *
water_file = r"C:\Users\dell\OneDrive\file\nc\water.nc"
list_path_sw = [pathf_part3_sw_ohigh, pathf_part3_sw_olow, pathf_part3_sw_o2020, pathf_part3_sw_obase, pathf_part3_sw_ous]

for path_sw in list_path_sw:
    only_water_nc(water_file, path_sw + 'pfas', path_sw + 'pfas_use')
    only_water_nc(water_file, path_sw + 'raw', path_sw + 'raw_use')

处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！


In [16]:
from path_config import *
water_file = r"C:\Users\dell\OneDrive\file\nc\water.nc"

list_path_lr = [pathf_part3_lr_o2020, pathf_part3_lr_obase, pathf_part3_lr_ohigh, pathf_part3_lr_olow, pathf_part3_lr_ous]
#list_path_lr = [pathf_part3_lr_o2020, pathf_part3_lr_obase, pathf_part3_lr_ohigh]

for path_lr in list_path_lr:
    only_water_nc(water_file, path_lr + 'pfas', path_lr + 'pfas_use')
    only_water_nc(water_file, path_lr + 'raw', path_lr + 'raw_use')

处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！
处理完成！


### 处理后使用PFAS

In [17]:
import os
import xarray as xr
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed

def process_single_nc_file(nc_file, output_path, dict_treat, 
                           list_pfas_all_safe, list_pfas_all_lc_safe, list_pfas_all_sc_safe):
    """
    处理单个nc文件
    
    Parameters:
    -----------
    nc_file : Path
        nc文件路径
    output_path : str
        输出路径
    dict_treat : dict
        PFAS名称与其开始使用年份的字典
    list_pfas_all_safe : list
        所有PFAS的安全名称列表
    list_pfas_all_lc_safe : list
        长链PFAS的安全名称列表
    list_pfas_all_sc_safe : list
        短链PFAS的安全名称列表
    
    Returns:
    --------
    str : 处理结果信息
    """
    try:
        ds = xr.open_dataset(nc_file)
        
        result_msg = f"处理文件: {nc_file.name}\n"

        for pfas_name, start_year in dict_treat.items():
            var_name = pfas_name  

            if var_name in ds.data_vars:
                result_msg += f"  处理变量 {var_name}, 开始使用年份: {start_year}\n"

                mask = ds['year'] < start_year
                ds[var_name] = ds[var_name].where(~mask, 0)
            else:
                result_msg += f"  警告: 变量 {var_name} 不在数据集中\n"
        
        result_msg += "  重新计算 lc_value\n"
        lc_vars = [var for var in list_pfas_all_lc_safe if var in ds.data_vars]
        if lc_vars:
            ds['lc_value'] = sum([ds[var] for var in lc_vars])
        else:
            result_msg += "    警告: 未找到长链PFAS变量\n"
        
        result_msg += "  重新计算 sc_value\n"
        sc_vars = [var for var in list_pfas_all_sc_safe if var in ds.data_vars]
        if sc_vars:
            ds['sc_value'] = sum([ds[var] for var in sc_vars])
        else:
            result_msg += "    警告: 未找到短链PFAS变量\n"
        
        result_msg += "  重新计算 value\n"
        all_vars = [var for var in list_pfas_all_safe if var in ds.data_vars]
        if all_vars:
            ds['value'] = sum([ds[var] for var in all_vars])
        else:
            result_msg += "    警告: 未找到PFAS变量\n"

        output_file = os.path.join(output_path, nc_file.name)
        ds.to_netcdf(output_file)
        result_msg += f"  保存至: {output_file}\n"
        
        ds.close()
        
        return result_msg
        
    except Exception as e:
        return f"处理文件 {nc_file.name} 时出错: {str(e)}\n"


def process_nc_files_parallel(input_path, output_path, dict_treat, 
                               list_pfas_all_safe, list_pfas_all_lc_safe, 
                               list_pfas_all_sc_safe, n_jobs=10):
    """
    并行处理nc文件：对特定PFAS在开始使用年份之前的数值归零，并重新计算汇总变量
    
    Parameters:
    -----------
    input_path : str
        输入nc文件所在路径
    output_path : str
        输出nc文件路径
    dict_treat : dict
        PFAS名称与其开始使用年份的字典，例如 {'HFPO-DA': 2009, 'ADONA': 2008}
    list_pfas_all_safe : list
        所有PFAS的安全名称列表
    list_pfas_all_lc_safe : list
        长链PFAS的安全名称列表
    list_pfas_all_sc_safe : list
        短链PFAS的安全名称列表
    n_jobs : int
        并行处理的进程数，默认为10
    """
    os.makedirs(output_path, exist_ok=True)
    nc_files = list(Path(input_path).glob('*.nc'))
    
    if not nc_files:
        print(f"在路径 {input_path} 中未找到nc文件")
        return
    
    print(f"找到 {len(nc_files)} 个nc文件")
    print(f"使用 {n_jobs} 个进程进行并行处理...")
    print("-" * 60)

    results = Parallel(n_jobs=n_jobs, verbose=10, backend='loky')(
        delayed(process_single_nc_file)(
            nc_file, output_path, dict_treat,
            list_pfas_all_safe, list_pfas_all_lc_safe, list_pfas_all_sc_safe
        ) for nc_file in nc_files
    )

    print("\n" + "=" * 60)
    print("处理结果详情:")
    print("=" * 60)
    for result in results:
        print(result)
    
    print("=" * 60)
    print(f"处理完成！所有 {len(nc_files)} 个文件已保存至 {output_path}")





In [18]:

if __name__ == "__main__":
    input_path = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_use"
    output_path = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final"
    
    process_nc_files_parallel(
        input_path, 
        output_path, 
        dict_treat, 
        list_pfas_all_safe, 
        list_pfas_all_lc_safe, 
        list_pfas_all_sc_safe,
        n_jobs=10
    )

找到 100 个nc文件
使用 10 个进程进行并行处理...
------------------------------------------------------------


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed:  2.5min
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed:  4.7min
[Parallel(n_jobs=10)]: Done  21 tasks      | elapsed:  6.5min
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:  6.9min
[Parallel(n_jobs=10)]: Done  41 tasks      | elapsed: 10.2min
[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed: 12.4min
[Parallel(n_jobs=10)]: Done  65 tasks      | elapsed: 15.2min
[Parallel(n_jobs=10)]: Done  78 tasks      | elapsed: 17.7min
[Parallel(n_jobs=10)]: Done  92 out of 100 | elapsed: 20.5min remaining:  1.8min



处理结果详情:
处理文件: sw_forecast_10322258.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  处理变量 ADONA, 开始使用年份: 2008
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final\sw_forecast_10322258.nc

处理文件: sw_forecast_10368276.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  处理变量 ADONA, 开始使用年份: 2008
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final\sw_forecast_10368276.nc

处理文件: sw_forecast_10543914.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  处理变量 ADONA, 开始使用年份: 2008
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final\sw_forecast_10543914.nc

处理文件: sw_forecast_10720120.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  处理变量 ADONA, 开始使用年份: 2008
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final\sw_forecast_10720120.nc

处理文件: sw_forecast_1110962.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  处理变量 ADONA, 开始使用年份: 2008
  重新计算

[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed: 21.5min finished


In [19]:

if __name__ == "__main__":
    input_path = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_use"
    output_path = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final"
    
    process_nc_files_parallel(
        input_path, 
        output_path, 
        dict_treat, 
        list_pfas_all_safe, 
        list_pfas_all_lc_safe, 
        list_pfas_all_sc_safe,
        n_jobs=10
    )

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


找到 100 个nc文件
使用 10 个进程进行并行处理...
------------------------------------------------------------


[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed:   52.6s
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed:  1.7min
[Parallel(n_jobs=10)]: Done  21 tasks      | elapsed:  2.4min
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:  2.6min
[Parallel(n_jobs=10)]: Done  41 tasks      | elapsed:  3.9min
[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed:  4.8min
[Parallel(n_jobs=10)]: Done  65 tasks      | elapsed:  5.8min
[Parallel(n_jobs=10)]: Done  78 tasks      | elapsed:  6.8min
[Parallel(n_jobs=10)]: Done  92 out of 100 | elapsed:  8.1min remaining:   41.9s



处理结果详情:
处理文件: lr_forecast_10720120_0.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  警告: 变量 ADONA 不在数据集中
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final\lr_forecast_10720120_0.nc

处理文件: lr_forecast_10720120_1.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  警告: 变量 ADONA 不在数据集中
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final\lr_forecast_10720120_1.nc

处理文件: lr_forecast_10720120_2.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  警告: 变量 ADONA 不在数据集中
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final\lr_forecast_10720120_2.nc

处理文件: lr_forecast_10720120_3.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  警告: 变量 ADONA 不在数据集中
  重新计算 lc_value
  重新计算 sc_value
  重新计算 value
  保存至: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final\lr_forecast_10720120_3.nc

处理文件: lr_forecast_10720120_4.nc
  处理变量 HFPO-DA, 开始使用年份: 2009
  警告: 变量 ADONA 不在数据集中
  重新计算 lc_va

[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:  8.3min finished


### 获取pfas

In [20]:
dir_lr_raw_2020_f = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final"
dir_lr_pfas_2020_f = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\pfas_final"

dir_sw_raw_2020_f  = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final"  
dir_sw_pfas_2020_f = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\pfas_final"

In [21]:
from joblib import Parallel, delayed



tasks = [
    (dir_sw_raw_2020_f, dir_sw_pfas_2020_f),
]

Parallel(n_jobs=10)(
    delayed(mg.from_seed_get_pfas_dask)(
        dir_seed=dir_seed,
        dir_pfas=dir_pfas,
        str_describe="sw",
        chunk_seed=1,    
        chunk_lat=60,    
        chunk_lon=120,
        approx=True
    )
    for dir_seed, dir_pfas in tasks
)


[None]

In [22]:
from joblib import Parallel, delayed

params = [
    (dir_lr_raw_2020_f, dir_lr_pfas_2020_f),
]

Parallel(n_jobs=10)(
    delayed(mg.from_seed_get_pfas_dask)(
        dir_seed=seed_dir,
        dir_pfas=pfas_dir,
        str_describe="lr",
        chunk_seed=1,   
        chunk_lat=60,   
        chunk_lon=120,
        approx=True
    )
    for seed_dir, pfas_dir in params
)


[None]

### 统计
下面的没优化  
优化的版本直接放fig里面去  
forecast stats ipynb  
没优化成功= =

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import xarray as xr

# ── 配置路径 ──────────────────────────────────────────
POP_MARK_PATH = r"E:\wyy\SPDB_database\data\raw\1\pop_mark.nc"
GEO_CLF_CSV = r"C:\Users\dell\OneDrive\file\csv\geo_clf_updated.csv"


def load_geo_classification_masks(csv_path):
    """
    读取 geo_clf_grid.csv，并构建 continent / development 的二维掩膜。
    返回：
        continent_masks_raw: dict[str, xr.DataArray]
        development_masks_raw: dict[str, xr.DataArray]
    """
    df = pd.read_csv(csv_path)

    required_cols = {'lon', 'lat', 'continent', 'development'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"{csv_path} 必须包含列: {required_cols}")

    valid_continents = ['Africa', 'Asia', 'Europe', 'Latin America', 'North America']
    valid_development = ['Developing regions', 'Developed regions']

    # 先做基础网格，避免重复 pivot 结构不一致
    grid = df[['lat', 'lon']].drop_duplicates()
    lat_vals = np.sort(grid['lat'].unique())
    lon_vals = np.sort(grid['lon'].unique())

    continent_masks_raw = {}
    for cont in valid_continents:
        tmp = df[['lat', 'lon', 'continent']].copy()
        tmp['mask'] = (tmp['continent'] == cont).astype(np.int8)
        pivot = tmp.pivot(index='lat', columns='lon', values='mask')
        pivot = pivot.reindex(index=lat_vals, columns=lon_vals).fillna(0).astype(np.int8)

        continent_masks_raw[cont] = xr.DataArray(
            pivot.values,
            coords={'lat': lat_vals, 'lon': lon_vals},
            dims=('lat', 'lon'),
            name=cont
        )

    development_masks_raw = {}
    for dev in valid_development:
        tmp = df[['lat', 'lon', 'development']].copy()
        tmp['mask'] = (tmp['development'] == dev).astype(np.int8)
        pivot = tmp.pivot(index='lat', columns='lon', values='mask')
        pivot = pivot.reindex(index=lat_vals, columns=lon_vals).fillna(0).astype(np.int8)

        development_masks_raw[dev] = xr.DataArray(
            pivot.values,
            coords={'lat': lat_vals, 'lon': lon_vals},
            dims=('lat', 'lon'),
            name=dev
        )

    return continent_masks_raw, development_masks_raw


def weighted_mean_timeseries(data_3d, mask_2d, weights_2d):
    """
    对 3D 数组 data_3d(year, lat, lon) 在 mask_2d 范围内计算逐年面积加权均值。
    返回 shape=(year,)
    """
    valid = mask_2d[None, :, :] & np.isfinite(data_3d)
    w = np.where(valid, weights_2d[None, :, :], 0.0)
    d = np.where(valid, data_3d, 0.0)

    denom = w.sum(axis=(1, 2))
    numer = (d * w).sum(axis=(1, 2))

    out = np.full(data_3d.shape[0], np.nan, dtype=float)
    good = denom > 0
    out[good] = numer[good] / denom[good]
    return out


def median_timeseries(data_3d, mask_2d):
    """
    对 3D 数组 data_3d(year, lat, lon) 在 mask_2d 范围内计算逐年中位数。
    返回 shape=(year,)
    """
    masked = np.where(mask_2d[None, :, :], data_3d, np.nan)
    return np.nanmedian(masked, axis=(1, 2))


def process_nc_files(input_dir, output_dir):
    """
    批量读取 NetCDF 文件，计算：
      - 全球均值（面积加权）与中位数
      - 有人区均值（面积加权）与中位数
      - 无人区均值（面积加权）与中位数
      - 各大洲均值（面积加权）与中位数
      - 发达/发展中地区均值（面积加权）与中位数
    并保存为 CSV。
    """

    # ── 读取 pop_mark ──────────────────────────────
    ds_pop = xr.open_dataset(POP_MARK_PATH)
    pop_mark_raw = ds_pop['population']
    ds_pop.close()

    # ── 读取地理分类 CSV ───────────────────────────
    continent_masks_raw, development_masks_raw = load_geo_classification_masks(GEO_CLF_CSV)

    results = []
    nc_files = glob.glob(os.path.join(input_dir, '*.nc'))

    continent_col_map = {
        'Africa': 'africa',
        'Asia': 'asia',
        'Europe': 'europe',
        'Latin America': 'latin_america',
        'North America': 'north_america'
    }
    development_col_map = {
        'Developing regions': 'developing',
        'Developed regions': 'developed'
    }

    for nc_path in nc_files:
        fname = os.path.basename(nc_path)

        if fname.startswith('sw_forecast'):
            seed_match = re.search(r'sw_forecast_(\d+)', fname)
            seed = seed_match.group(1) if seed_match else 'unknown'
        elif fname.startswith('lr_forecast'):
            seed_match = re.search(r'lr_forecast_(\d+_\d+)', fname)
            seed = seed_match.group(1) if seed_match else 'unknown'
        else:
            seed = 'unknown'

        ds = xr.open_dataset(nc_path)

        if 'lat' not in ds.coords or 'lon' not in ds.coords:
            ds.close()
            raise ValueError(f"{fname} 不包含坐标 lat/lon")
        if 'year' not in ds.coords:
            ds.close()
            raise ValueError(f"{fname} 不包含坐标 year")

        # 数据变量
        num_vars = list(ds.data_vars)

        years = ds['year'].values
        lat = ds['lat'].values
        lon = ds['lon'].values

        # ── 一次性对齐各种掩膜到当前格网 ──────────────────
        pop_mark = pop_mark_raw.interp(
            lat=ds['lat'], lon=ds['lon'], method='nearest'
        )
        inhabited_mask = (pop_mark == 1).values
        uninhabited_mask = (pop_mark == 0).values
        global_mask = np.ones((len(lat), len(lon)), dtype=bool)

        continent_masks = {}
        for cont, da in continent_masks_raw.items():
            continent_masks[cont] = (
                da.interp(lat=ds['lat'], lon=ds['lon'], method='nearest').values == 1
            )

        development_masks = {}
        for dev, da in development_masks_raw.items():
            development_masks[dev] = (
                da.interp(lat=ds['lat'], lon=ds['lon'], method='nearest').values == 1
            )

        # ── 纬度权重，只算一次 ──────────────────────────
        lat_weights = np.cos(np.deg2rad(lat)).astype(float)   # (lat,)
        weights_2d = np.broadcast_to(lat_weights[:, None], (len(lat), len(lon)))

        # ── 汇总所有区域 mask，统一计算 ──────────────────
        region_masks = {
            'g': global_mask,
            'ih': inhabited_mask,
            'uih': uninhabited_mask,
        }

        for cont, short_name in continent_col_map.items():
            region_masks[short_name] = continent_masks[cont]

        for dev, short_name in development_col_map.items():
            region_masks[short_name] = development_masks[dev]

        # ── 逐变量处理，但每个变量一次性处理所有 year ───────
        for var in num_vars:
            data = ds[var]

            # 跳过非(year, lat, lon)变量
            if not all(dim in data.dims for dim in ['year', 'lat', 'lon']):
                continue

            # 统一维度顺序
            data = data.transpose('year', 'lat', 'lon')
            data_3d = data.values  # shape = (year, lat, lon)

            stats = {}
            for region_name, mask_2d in region_masks.items():
                stats[f'{region_name}_mean'] = weighted_mean_timeseries(data_3d, mask_2d, weights_2d)
                stats[f'{region_name}_median'] = median_timeseries(data_3d, mask_2d)

            # 组装结果
            for i, y in enumerate(years):
                row = {
                    'pfas': var,
                    'seed': seed,
                    'year': int(y),

                    'g_mean': stats['g_mean'][i],
                    'g_median': stats['g_median'][i],

                    'ih_mean': stats['ih_mean'][i],
                    'ih_median': stats['ih_median'][i],

                    'uih_mean': stats['uih_mean'][i],
                    'uih_median': stats['uih_median'][i],

                    'africa_mean': stats['africa_mean'][i],
                    'africa_median': stats['africa_median'][i],

                    'asia_mean': stats['asia_mean'][i],
                    'asia_median': stats['asia_median'][i],

                    'europe_mean': stats['europe_mean'][i],
                    'europe_median': stats['europe_median'][i],

                    'latin_america_mean': stats['latin_america_mean'][i],
                    'latin_america_median': stats['latin_america_median'][i],

                    'north_america_mean': stats['north_america_mean'][i],
                    'north_america_median': stats['north_america_median'][i],

                    'developing_mean': stats['developing_mean'][i],
                    'developing_median': stats['developing_median'][i],

                    'developed_mean': stats['developed_mean'][i],
                    'developed_median': stats['developed_median'][i],
                }
                results.append(row)

        ds.close()

    df = pd.DataFrame(results)

    os.makedirs(output_dir, exist_ok=True)
    outfile = os.path.join(output_dir, 'global_stats_f.csv')
    df.to_csv(outfile, index=False, encoding='utf-8-sig')
    print(f"✅ 已保存结果到: {outfile}")

In [3]:
process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020"
)


✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_f.csv


In [4]:
process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020"
)


✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_f.csv


In [5]:


process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base"
)

process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_high\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_high"
)

process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_low\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_low"
)

process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_us\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_us"
)

✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\global_stats_f.csv
✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_high\global_stats_f.csv
✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_low\global_stats_f.csv
✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_us\global_stats_f.csv


In [6]:

process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base"
)

process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high"
)
process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low"
)
process_nc_files(
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us\raw_use",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us"
)

✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\global_stats_f.csv
✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high\global_stats_f.csv
✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low\global_stats_f.csv
✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us\global_stats_f.csv


### 进一步统计

In [8]:
import os
import numpy as np
import pandas as pd


def process_global_stats(file_path):
    """
    读取 global_stats.csv，按不同区域分别汇总，
    每个区域输出一个 summary 文件。

    汇总方式（按 pfas, year 分组）：
      - 对 mean 列：
          mean_mean, mean_std, mean_sem
          mean_ci95_low, mean_ci95_high   # 基于 SEM 的均值置信区间
          mean_q2_5, mean_q20, mean_q50, mean_q80, mean_q97_5  # 基于 seed 分布的经验分位数
      - 对 median 列：
          median_mean
          median_std
          median_median
          median_MAD
          median_q2_5, median_q20, median_q50, median_q80, median_q97_5
      - n_seeds
    """
    df = pd.read_csv(file_path)

    # 各区域的列映射：(输出文件后缀, mean列名, median列名)
    regions = [
        ("g", "g_mean", "g_median"),
        ("ih", "ih_mean", "ih_median"),
        ("uih", "uih_mean", "uih_median"),
        ("africa", "africa_mean", "africa_median"),
        ("asia", "asia_mean", "asia_median"),
        ("europe", "europe_mean", "europe_median"),
        ("latin_america", "latin_america_mean", "latin_america_median"),
        ("north_america", "north_america_mean", "north_america_median"),
        ("developing", "developing_mean", "developing_median"),
        ("developed", "developed_mean", "developed_median"),
    ]

    def safe_std(x):
        return np.std(x, ddof=1) if len(x) > 1 else np.nan

    def safe_sem(x):
        if len(x) > 1:
            std = np.std(x, ddof=1)
            return std / np.sqrt(len(x))
        return np.nan

    def safe_mad(x):
        if len(x) == 0:
            return np.nan
        med = np.median(x)
        return np.median(np.abs(x - med))

    def agg_func(mean_col, median_col):
        def _agg(group):
            mean_vals = group[mean_col].dropna().values

            # mean 统计
            if len(mean_vals) > 0:
                mean_mean = np.mean(mean_vals)
                mean_std = safe_std(mean_vals)
                mean_sem = safe_sem(mean_vals)
                mean_q2_5 = np.percentile(mean_vals, 2.5)
                mean_q20 = np.percentile(mean_vals, 20)
                mean_q50 = np.percentile(mean_vals, 50)
                mean_q80 = np.percentile(mean_vals, 80)
                mean_q97_5 = np.percentile(mean_vals, 97.5)
            else:
                mean_mean = np.nan
                mean_std = np.nan
                mean_sem = np.nan
                mean_q2_5 = np.nan
                mean_q20 = np.nan
                mean_q50 = np.nan
                mean_q80 = np.nan
                mean_q97_5 = np.nan


            return pd.Series({
                "mean_mean": mean_mean,
                "mean_std": mean_std,
                "mean_sem": mean_sem,
                "mean_q2_5": mean_q2_5,
                "mean_q20": mean_q20,
                "mean_q50": mean_q50,
                "mean_q80": mean_q80,
                "mean_q97_5": mean_q97_5,
                "n_mean_seeds": len(mean_vals),
            })
        return _agg

    base_dir = os.path.dirname(file_path)

    for suffix, mean_col, median_col in regions:
        required = {"pfas", "seed", "year", mean_col, median_col}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(f"区域 {suffix} 缺少必要列: {missing}")

        result_df = (
            df.groupby(["pfas", "year"], sort=False)
              .apply(agg_func(mean_col, median_col))
              .reset_index()
        )

        out_path = os.path.join(base_dir, f"global_stats_{suffix}.csv")
        result_df.to_csv(out_path, index=False, float_format="%.6f")
        print(f"✅ 已保存: {out_path}")

In [9]:
# 使用示例：
paths = [
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_low\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_high\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_us\global_stats_f.csv",
]
for p in paths:
    process_global_stats(p)

✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_g.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_ih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_uih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_africa.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_asia.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_europe.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_latin_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_north_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_developing.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_developed.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\global_stats_g.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2050_base\global_stats_ih.

In [10]:
paths = [
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_low\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_high\global_stats_f.csv",
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_us\global_stats_f.csv",
]
for p in paths:
    process_global_stats(p)

✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_g.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_ih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_uih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_africa.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_asia.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_europe.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_latin_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_north_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_developing.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_developed.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\global_stats_g.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2050_base\global_stats_ih.